# 06 — Media Mix Model

Synthetic MMM notebook with adstock, saturation and budget response simulation.

**Project:** Marketing Analytics Causal & LTV Lab  
**Style:** Hands-on, advanced, interview-ready notebook  
**How to use:** Run cell by cell, inspect outputs, then discuss interpretation and pitfalls.


## Main notebook code

Run this notebook and then we will discuss the output, assumptions and pitfalls.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
RANDOM_STATE=42; rng=np.random.default_rng(RANDOM_STATE); SYNTHETIC_DIR=Path('../data/synthetic'); PROCESSED_DIR=Path('../data/processed'); SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True); PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
def adstock(x,decay):
    out=np.zeros_like(x,dtype=float); out[0]=x[0]
    for i in range(1,len(x)): out[i]=x[i]+decay*out[i-1]
    return out
def hill(x,a,g): return (x**a)/(x**a+g**a)
weeks=pd.date_range('2023-01-01',periods=156,freq='W'); t=np.arange(len(weeks)); m=pd.DataFrame({'week':weeks,'seasonality':np.sin(2*np.pi*t/52),'trend':t})
base=rng.gamma(5,1000,len(weeks)); m['search_spend']=base; m['social_spend']=base*.55+rng.gamma(4,700,len(weeks)); m['display_spend']=rng.gamma(3,500,len(weeks)); m['email_spend']=rng.gamma(2,200,len(weeks)); m['promotion']=rng.binomial(1,.15,len(weeks)); m['competitor_index']=rng.normal(100,10,len(weeks))
effects=2200*hill(adstock(m.search_spend.values,.4),1.2,6000)+1600*hill(adstock(m.social_spend.values,.6),1.1,5000)+700*hill(adstock(m.display_spend.values,.3),1,2500)+500*hill(adstock(m.email_spend.values,.2),1,1200)
m['revenue']=60000+150*m.trend+3000*m.seasonality+5000*m.promotion-80*m.competitor_index+effects+rng.normal(0,2500,len(weeks))
for col,dec,a,g in [('search_spend',.4,1.2,6000),('social_spend',.6,1.1,5000),('display_spend',.3,1,2500),('email_spend',.2,1,1200)]: m[f'{col}_adstock_sat']=hill(adstock(m[col].values,dec),a,g)
features=['trend','seasonality','promotion','competitor_index','search_spend_adstock_sat','social_spend_adstock_sat','display_spend_adstock_sat','email_spend_adstock_sat']
split=int(len(m)*.8); scaler=StandardScaler(); Xtr=scaler.fit_transform(m.loc[:split-1,features]); Xte=scaler.transform(m.loc[split:,features]); ytr=m.loc[:split-1,'revenue']; yte=m.loc[split:,'revenue']
model=Ridge(alpha=10).fit(Xtr,ytr); pred=model.predict(Xte); print('MAE',mean_absolute_error(yte,pred),'RMSE',mean_squared_error(yte,pred,squared=False),'R2',r2_score(yte,pred)); display(pd.DataFrame({'feature':features,'coef':model.coef_}).sort_values('coef',ascending=False))
plt.figure(figsize=(10,4)); plt.plot(m.loc[split:,'week'],yte,label='actual'); plt.plot(m.loc[split:,'week'],pred,label='predicted'); plt.legend(); plt.title('MMM Revenue Forecast'); plt.show()
m.to_csv(SYNTHETIC_DIR/'synthetic_mmm_weekly.csv',index=False); m.to_csv(PROCESSED_DIR/'mmm_features.csv',index=False)


## Discussion prompts

1. What assumption is strongest here?
2. Which pitfall would break the conclusion?
3. How would you explain this to a non-technical stakeholder?
